# Auth0 Management API Ingestion

Pulls Auth0 tenant data (users, clients, connections, logs) into the HillStreet warehouse via the machine-to-machine `client_credentials` flow. All credentials are read from the Databricks secrets scope `hillstreet_auth0` — nothing is hardcoded here.

**Prerequisite (one-time, Auth0 dashboard):** Applications → *Hillstreet — Composio Auth0* must be authorized on the **Auth0 Management API** (Applications → APIs → Auth0 Management API → Machine to Machine tab) with at least `read:users`, `read:clients`, `read:connections`, `read:logs`, `read:tenant_settings`. Without this grant the tenant issues tokens with no scopes and every `/api/v2/` call returns 401.

In [0]:
# Configuration — all values from the hillstreet_auth0 secrets scope
SCOPE = "hillstreet_auth0"
AUTH0_DOMAIN = dbutils.secrets.get(SCOPE, "auth0_domain")          # dev-yd7je5twh6ug2h5v.us.auth0.com
CLIENT_ID = dbutils.secrets.get(SCOPE, "auth0_client_id")
CLIENT_SECRET = dbutils.secrets.get(SCOPE, "auth0_client_secret")
AUDIENCE = dbutils.secrets.get(SCOPE, "auth0_audience")            # .../api/v2/

OUTPUT_VOLUME = "/Volumes/hillstreet/warehouse/data/auth0"
ENTITIES = ["users", "clients", "connections", "logs"]
print(f"Configured for tenant: {AUTH0_DOMAIN} | entities: {ENTITIES} | output: {OUTPUT_VOLUME}")

In [0]:
# Mint a fresh M2M token and verify it carries scopes before making API calls
import requests, json, base64

def get_access_token():
    r = requests.post(
        f"https://{AUTH0_DOMAIN}/oauth/token",
        json={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "audience": AUDIENCE,
            "grant_type": "client_credentials",
        },
        timeout=30,
    )
    r.raise_for_status()
    token = r.json()["access_token"]
    payload = token.split(".")[1]
    claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
    if not claims.get("scope"):
        raise PermissionError(
            "Token issued without scopes — authorize the M2M app on the Auth0 Management API "
            "(see prerequisite in the title cell), then rerun."
        )
    return token

TOKEN = get_access_token()
print("M2M token minted with valid scopes.")

In [0]:
# Fetch each entity with pagination and write dated JSONL snapshots to the volume
from datetime import date
import time

SNAPSHOT_DATE = date.today().isoformat()

def fetch_all(entity, per_page=100, max_pages=100):
    """Paginate a Management API list endpoint. Logs use checkpoint-style pagination."""
    results, params, page = [], {"per_page": per_page, "page": 0}, 0
    while page < max_pages:
        params["page"] = page
        r = requests.get(f"https://{AUTH0_DOMAIN}/api/v2/{entity}",
                         headers={"Authorization": f"Bearer {TOKEN}"}, params=params, timeout=30)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        results.extend(batch)
        if len(batch) < per_page:
            break
        page += 1
        time.sleep(0.2)  # stay under Auth0 rate limits
    return results

for entity in ENTITIES:
    rows = fetch_all(entity)
    path = f"{OUTPUT_VOLUME}/{entity}/{SNAPSHOT_DATE}.jsonl"
    dbutils.fs.mkdirs(f"{OUTPUT_VOLUME}/{entity}")
    with dbutils.fs.put(path, "\n".join(json.dumps(row, default=str) for row in rows), overwrite=True) as f:
        pass
    print(f"{entity}: {len(rows)} rows -> {path}")

In [0]:
# Read the latest snapshot back with Spark and summarize
latest = {}
for entity in ENTITIES:
    files = dbutils.fs.ls(f"{OUTPUT_VOLUME}/{entity}")
    if files:
        latest[entity] = sorted(files, key=lambda x: x.name)[-1].path

for entity, path in latest.items():
    df = spark.read.json(path)
    print(f"{entity}: {df.count()} rows | columns: {df.columns[:8]}")
    display(df.limit(3))